# TGN 从零实现：交易事件流中的快速重复识别

## 面试问题

面试时我会把 TGN 拆成事件消息、时间编码、消息聚合、节点记忆更新和下游预测。预测当前事件时只能读取此前记忆，然后才能用当前事件更新源/目标节点，避免未来泄漏。时间差需要基于每个节点最后一次事件计算，并对输入事件按时间排序。记忆更新可用 GRUCell，消息由源记忆、目标记忆、事件特征和时间编码共同生成。训练时可在短序列上反向传播，长事件流通常截断梯度并把 memory 当外部状态管理。下面用十二笔脱敏交易比较金额阈值，手写时间编码、消息与双端记忆更新，并展示事件账本和乱序失败。

## 真实案例

事件流包含三名用户、三家商户和十二笔按分钟排序的交易。标签表示同一用户—商户在短时间内快速重复，金额可能很小；两笔大额正常交易故意让金额规则误报。数据是结构真实的离线教学序列，省略设备、币种和调查延迟。

本实验是用于解释机制的确定性小样本，所有指标均标记为“教学实验”，不能外推为线上收益。

In [1]:
import math  # 导入圆周率用于可学习余弦时间编码。
import torch  # 导入 PyTorch 以实现时序记忆与训练。
from torch import nn  # 导入 GRUCell 和线性层等基础模块。
import torch.nn.functional as F  # 导入 one-hot、激活与二元交叉熵。
torch.manual_seed(50)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定短序列训练。
user_ids = ["用户-A", "用户-B", "用户-C"]  # 定义三名脱敏用户节点。
merchant_ids = ["便利店", "数码店", "咖啡店"]  # 定义三家可读商户节点。
events = [(0, 0, 0, 30.0, 0), (4, 1, 1, 150.0, 0), (8, 0, 0, 32.0, 1), (15, 2, 2, 25.0, 0), (18, 2, 0, 28.0, 0), (20, 1, 1, 155.0, 0), (22, 0, 0, 31.0, 0), (30, 1, 2, 22.0, 0), (31, 1, 2, 21.0, 1), (40, 2, 0, 29.0, 0), (43, 2, 0, 27.0, 1), (50, 0, 1, 180.0, 0)]  # 保存时间、用户、商户、金额和快速重复标签。
print("序号  时间  用户    商户    金额   快速重复标签")  # 打印事件流输入表头。
for event_index, event in enumerate(events):  # 按时间逐事件展示原始记录。
    timestamp, user_index, merchant_index, amount, label = event  # 解包当前交易的五个字段。
    print(f"{event_index:02d}    {timestamp:>2}   {user_ids[user_index]}  {merchant_ids[merchant_index]:<4}  {amount:>6.1f}  {label}")  # 输出当前交易的可读字段。
print(f"事件数={len(events)}，时间范围={events[0][0]}–{events[-1][0]} 分钟，正例数={sum(event[4] for event in events)}")  # 汇总事件流规模与标签分布。

序号  时间  用户    商户    金额   快速重复标签
00     0   用户-A  便利店     30.0  0
01     4   用户-B  数码店    150.0  0
02     8   用户-A  便利店     32.0  1
03    15   用户-C  咖啡店     25.0  0
04    18   用户-C  便利店     28.0  0
05    20   用户-B  数码店    155.0  0
06    22   用户-A  便利店     31.0  0
07    30   用户-B  咖啡店     22.0  0
08    31   用户-B  咖啡店     21.0  1
09    40   用户-C  便利店     29.0  0
10    43   用户-C  便利店     27.0  1
11    50   用户-A  数码店    180.0  0
事件数=12，时间范围=0–50 分钟，正例数=3


## 基线：只按金额大于 100 判风险

金额规则看不到快速重复，因此漏掉三笔小额异常，并误报两笔大额正常交易。它和 TGN 在相同十二笔事件上计算准确率。

In [2]:
event_labels = torch.tensor([event[4] for event in events], dtype=torch.float32)  # 提取十二笔交易的二元标签。
baseline_predictions = torch.tensor([int(event[3] > 100.0) for event in events], dtype=torch.float32)  # 用固定金额阈值产生基线预测。
baseline_accuracy = (baseline_predictions == event_labels).float().mean().item()  # 计算金额基线准确率。
print("序号  金额   基线预测  真实标签")  # 打印逐事件基线结果表头。
for event_index, event in enumerate(events):  # 遍历事件观察金额规则误报与漏报。
    print(f"{event_index:02d}    {event[3]:>6.1f}  {int(baseline_predictions[event_index].item())}         {event[4]}")  # 输出当前金额、规则预测与真实标签。
print(f"金额阈值基线准确率={baseline_accuracy:.1%}")  # 汇总同数据上的基线指标。

序号  金额   基线预测  真实标签
00      30.0  0         0
01     150.0  1         0
02      32.0  0         1
03      25.0  0         0
04      28.0  0         0
05     155.0  1         0
06      31.0  0         0
07      22.0  0         0
08      21.0  0         1
09      29.0  0         0
10      27.0  0         1
11     180.0  1         0
金额阈值基线准确率=50.0%


## 手写核心：时间编码、事件消息和双端 GRU 记忆

每个事件先用更新前的用户/商户 memory 预测，再构造消息更新两端。`run_event_stream` 使用 one-hot 门生成新的 memory 张量，保持短序列中的梯度链路，不调用现成 TGN 框架。

In [3]:
class TimeEncoder(nn.Module):  # 定义把连续时间差映射到多频特征的模块。
    def __init__(self, time_dim):  # 根据时间编码维度创建可学习频率和相位。
        super().__init__()  # 初始化父类以注册参数。
        initial_frequencies = torch.logspace(0.0, -2.0, steps=time_dim)  # 初始化从快到慢的时间频率。
        self.frequencies = nn.Parameter(initial_frequencies)  # 注册可学习频率参数。
        self.phases = nn.Parameter(torch.zeros(time_dim))  # 注册可学习相位参数。
    def forward(self, delta_time):  # 把归一化时间差编码成余弦向量。
        return torch.cos(delta_time.unsqueeze(-1) * self.frequencies + self.phases)  # 输出多频连续时间表示。
class TGNCell(nn.Module):  # 定义单事件预测与双端记忆更新单元。
    def __init__(self, memory_dim=8, time_dim=4):  # 创建时间编码、预测头、消息网络和 GRU 更新器。
        super().__init__()  # 初始化父类以注册全部参数。
        self.time_encoder = TimeEncoder(time_dim)  # 创建可学习的时间差编码器。
        context_dim = memory_dim * 2 + time_dim * 2 + 1  # 计算源记忆、目标记忆、双时间和金额的拼接维度。
        self.score_hidden = nn.Linear(context_dim, 16)  # 创建当前事件风险预测隐藏层。
        self.score_output = nn.Linear(16, 1)  # 输出当前事件的二元风险 logit。
        self.message_hidden = nn.Linear(context_dim, memory_dim)  # 把事件上下文压缩为记忆消息。
        self.user_updater = nn.GRUCell(memory_dim, memory_dim)  # 用 GRUCell 更新用户节点记忆。
        self.merchant_updater = nn.GRUCell(memory_dim, memory_dim)  # 用独立 GRUCell 更新商户节点记忆。
    def forward(self, user_memory, merchant_memory, amount_feature, user_delta, merchant_delta):  # 在更新前记忆上预测并生成新记忆。
        user_time = self.time_encoder(user_delta)  # 编码用户距离上次事件的时间差。
        merchant_time = self.time_encoder(merchant_delta)  # 编码商户距离上次事件的时间差。
        context = torch.cat([user_memory, merchant_memory, amount_feature.view(1), user_time, merchant_time], dim=0)  # 拼接当前事件的全部历史与属性证据。
        score_hidden = torch.relu(self.score_hidden(context))  # 提取当前事件的风险预测表示。
        logit = self.score_output(score_hidden).squeeze(0)  # 在更新记忆前输出风险 logit。
        message = torch.tanh(self.message_hidden(context))  # 生成写入节点记忆的有界事件消息。
        updated_user = self.user_updater(message.unsqueeze(0), user_memory.unsqueeze(0)).squeeze(0)  # 用当前消息更新源用户记忆。
        updated_merchant = self.merchant_updater(message.unsqueeze(0), merchant_memory.unsqueeze(0)).squeeze(0)  # 用当前消息更新目标商户记忆。
        return logit, updated_user, updated_merchant, message, user_time, merchant_time  # 返回预测和全部时序中间量。
def run_event_stream(model, event_records, collect_ledger=False):  # 按时间顺序执行完整 TGN 事件流。
    user_memories = torch.zeros(len(user_ids), 8)  # 为所有用户初始化零记忆。
    merchant_memories = torch.zeros(len(merchant_ids), 8)  # 为所有商户初始化零记忆。
    last_user_times = {}  # 保存每个用户最近一次事件时间。
    last_merchant_times = {}  # 保存每个商户最近一次事件时间。
    logits = []  # 保存每个事件在更新前得到的预测 logit。
    ledger = []  # 保存可读的记忆更新账本。
    for event_index, event in enumerate(event_records):  # 严格按输入顺序处理每一笔交易。
        timestamp, user_index, merchant_index, amount, label = event  # 解包当前事件字段。
        user_delta_value = timestamp - last_user_times.get(user_index, timestamp)  # 计算用户距离上次事件的分钟差。
        merchant_delta_value = timestamp - last_merchant_times.get(merchant_index, timestamp)  # 计算商户距离上次事件的分钟差。
        user_delta = torch.tensor(float(user_delta_value) / 10.0)  # 把用户时间差缩放到稳定数值范围。
        merchant_delta = torch.tensor(float(merchant_delta_value) / 10.0)  # 把商户时间差缩放到稳定数值范围。
        amount_feature = torch.tensor(float(amount) / 200.0)  # 把交易金额缩放到零至一附近。
        old_user_memory = user_memories[user_index]  # 读取更新前的用户节点记忆。
        old_merchant_memory = merchant_memories[merchant_index]  # 读取更新前的商户节点记忆。
        logit, new_user_memory, new_merchant_memory, message, user_time, merchant_time = model(old_user_memory, old_merchant_memory, amount_feature, user_delta, merchant_delta)  # 先预测当前事件再生成记忆更新。
        user_selector = F.one_hot(torch.tensor(user_index), num_classes=len(user_ids)).float().unsqueeze(1)  # 创建只选中当前用户的更新门。
        merchant_selector = F.one_hot(torch.tensor(merchant_index), num_classes=len(merchant_ids)).float().unsqueeze(1)  # 创建只选中当前商户的更新门。
        user_memories = user_memories * (1.0 - user_selector) + new_user_memory.unsqueeze(0) * user_selector  # 以可导的非原地方式写回当前用户记忆。
        merchant_memories = merchant_memories * (1.0 - merchant_selector) + new_merchant_memory.unsqueeze(0) * merchant_selector  # 以可导方式写回当前商户记忆。
        last_user_times[user_index] = timestamp  # 更新当前用户最近事件时间。
        last_merchant_times[merchant_index] = timestamp  # 更新当前商户最近事件时间。
        logits.append(logit)  # 保存当前事件更新前预测。
        if collect_ledger:  # 仅在评估时保存可读账本以避免训练开销。
            ledger.append((event_index, user_delta_value, merchant_delta_value, old_user_memory.norm().item(), new_user_memory.norm().item(), message.detach().clone(), user_time.detach().clone()))  # 记录时间差、记忆范数、消息和时间编码。
    return torch.stack(logits), ledger  # 返回完整事件流 logits 和可选账本。
tgn = TGNCell()  # 实例化手写时序图记忆单元。
print(tgn)  # 展示时间编码、消息网络与双 GRU 的实际结构。
print(f"可训练参数量={sum(parameter.numel() for parameter in tgn.parameters())}")  # 输出教学模型参数规模。

TGNCell(
  (time_encoder): TimeEncoder()
  (score_hidden): Linear(in_features=25, out_features=16, bias=True)
  (score_output): Linear(in_features=16, out_features=1, bias=True)
  (message_hidden): Linear(in_features=25, out_features=8, bias=True)
  (user_updater): GRUCell(8, 8)
  (merchant_updater): GRUCell(8, 8)
)
可训练参数量=1513


In [4]:
optimizer = torch.optim.Adam(tgn.parameters(), lr=0.02)  # 创建优化器更新时间、消息、记忆和预测参数。
loss_trace = []  # 保存完整事件流二元交叉熵轨迹。
first_time_gradient = 0.0  # 预留首轮时间频率梯度范数。
for epoch in range(501):  # 在十二事件短序列上执行五百零一次更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    sequence_logits, training_ledger = run_event_stream(tgn, events, collect_ledger=False)  # 从零记忆顺序执行完整事件流。
    loss = F.binary_cross_entropy_with_logits(sequence_logits, event_labels)  # 计算十二笔事件的二元分类损失。
    loss.backward()  # 沿事件顺序反向传播到时间编码和记忆更新。
    if epoch == 0:  # 首轮记录真实时间编码梯度规模。
        first_time_gradient = tgn.time_encoder.frequencies.grad.norm().item()  # 读取可学习频率的首轮梯度范数。
    torch.nn.utils.clip_grad_norm_(tgn.parameters(), max_norm=5.0)  # 裁剪长链路梯度以避免数值爆炸。
    optimizer.step()  # 根据当前梯度更新 TGN 参数。
    loss_trace.append(loss.item())  # 保存当前轮事件流损失。
    if epoch in [0, 50, 200, 500]:  # 选择关键轮次输出真实训练轨迹。
        current_accuracy = ((torch.sigmoid(sequence_logits) >= 0.5).float() == event_labels).float().mean().item()  # 计算当前事件分类准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} event_accuracy={current_accuracy:.1%}")  # 输出损失与准确率变化。
tgn.eval()  # 切换到评估模式生成稳定事件账本。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    final_logits, final_ledger = run_event_stream(tgn, events, collect_ledger=True)  # 从零记忆重放完整有序事件流。
tgn_probabilities = torch.sigmoid(final_logits)  # 把事件 logits 转成风险概率。
tgn_predictions = (tgn_probabilities >= 0.5).float()  # 用固定零点五阈值得到分类结果。
tgn_accuracy = (tgn_predictions == event_labels).float().mean().item()  # 计算 TGN 在同一事件流上的准确率。
print(f"首轮时间频率梯度范数={first_time_gradient:.6f}")  # 输出非零梯度证明时间编码被真实训练。
print(f"事件-08 消息向量前四维={[round(value, 3) for value in final_ledger[8][5][:4].tolist()]}")  # 展示一笔快速重复事件的消息中间量。
print(f"事件-08 时间编码={[round(value, 3) for value in final_ledger[8][6].tolist()]}")  # 展示同一事件的用户时间编码。

epoch=000 loss=0.6640 event_accuracy=75.0%


epoch=050 loss=0.0060 event_accuracy=100.0%


epoch=200 loss=0.0001 event_accuracy=100.0%


epoch=500 loss=0.0000 event_accuracy=100.0%
首轮时间频率梯度范数=0.011026
事件-08 消息向量前四维=[-1.0, 0.998, -1.0, -0.565]
事件-08 时间编码=[0.981, 0.975, 0.982, 0.997]


## 结果解读：预测发生在记忆更新之前

账本给出每笔交易的用户时间差、更新前后 memory 范数、风险概率和标签。快速重复事件应依赖此前记忆，而不是把当前标签写入状态后再预测。

In [5]:
print("序号  时间  用户-商户          Δ用户  memory范数前->后  概率   基线  TGN  标签")  # 打印逐事件时序结果表头。
for event_index, event in enumerate(events):  # 遍历十二笔事件展示时间与记忆证据。
    timestamp, user_index, merchant_index, amount, label = event  # 解包当前交易字段。
    ledger_row = final_ledger[event_index]  # 读取当前事件的记忆更新账本。
    pair_name = f"{user_ids[user_index]}-{merchant_ids[merchant_index]}"  # 构造可读用户商户对名称。
    print(f"{event_index:02d}    {timestamp:>2}   {pair_name:<14}  {ledger_row[1]:>3}    {ledger_row[3]:.3f}->{ledger_row[4]:.3f}       {tgn_probabilities[event_index]:.3f}   {int(baseline_predictions[event_index].item())}     {int(tgn_predictions[event_index].item())}    {label}")  # 输出当前事件完整时序预测结果。
print(f"同事件准确率：金额基线={baseline_accuracy:.1%}，手写 TGN={tgn_accuracy:.1%}")  # 汇总基线与 TGN 的同口径指标。

序号  时间  用户-商户          Δ用户  memory范数前->后  概率   基线  TGN  标签
00     0   用户-A-便利店          0    0.000->2.026       0.000   0     0    0
01     4   用户-B-数码店          0    0.000->1.778       0.000   1     0    0
02     8   用户-A-便利店          8    2.026->2.290       1.000   0     1    1
03    15   用户-C-咖啡店          0    0.000->2.048       0.000   0     0    0
04    18   用户-C-便利店          3    2.048->2.294       0.000   0     0    0
05    20   用户-B-数码店         16    1.778->2.047       0.000   1     0    0
06    22   用户-A-便利店         14    2.290->2.367       0.000   0     0    0
07    30   用户-B-咖啡店         10    2.047->2.257       0.000   0     0    0
08    31   用户-B-咖啡店          1    2.257->2.016       1.000   0     1    1
09    40   用户-C-便利店         22    2.294->2.548       0.000   0     0    0
10    43   用户-C-便利店          3    2.548->2.229       1.000   0     1    1
11    50   用户-A-数码店         28    2.367->2.222       0.000   1     0    0
同事件准确率：金额基线=50.0%，手写 TGN=100.0%


## 失败案例：文件到达顺序不等于事件时间顺序

下面交换时间 30 和 31 的同一用户事件。若直接按文件顺序更新 memory，第二条会得到负时间差，表示状态已经看过“未来”。修复是在进入 TGN 前按事件时间排序，并用稳定事件 ID 处理同时间事件。

In [6]:
out_of_order_events = events.copy()  # 复制原始有序事件流以构造乱序输入。
out_of_order_events[7], out_of_order_events[8] = out_of_order_events[8], out_of_order_events[7]  # 交换同一用户在时间三十和三十一的两笔交易。
def count_negative_user_deltas(event_records):  # 统计按输入顺序计算时出现的负用户时间差。
    last_times = {}  # 保存已经处理过的每个用户时间。
    negative_rows = []  # 保存出现负时间差的事件详情。
    for row_index, event in enumerate(event_records):  # 按传入顺序逐条检查时间单调性。
        timestamp, user_index, merchant_index, amount, label = event  # 解包当前事件字段。
        if user_index in last_times and timestamp - last_times[user_index] < 0:  # 检查当前用户时间是否倒退。
            negative_rows.append((row_index, user_ids[user_index], timestamp - last_times[user_index]))  # 记录负时间差位置、用户和数值。
        last_times[user_index] = timestamp  # 更新当前用户已处理时间。
    return negative_rows  # 返回全部时间倒退记录。
bad_negative_rows = count_negative_user_deltas(out_of_order_events)  # 检查错误文件顺序产生的负时间差。
sorted_events = sorted(out_of_order_events, key=lambda event: event[0])  # 按事件时间重新稳定排序输入。
fixed_negative_rows = count_negative_user_deltas(sorted_events)  # 检查排序后时间差是否恢复非负。
print(f"直接按文件顺序的负时间差记录={bad_negative_rows}")  # 展示状态读取未来造成的异常时间差。
print(f"按 event_time 排序后的负时间差记录={fixed_negative_rows}")  # 展示排序门禁消除时间倒退。
print("修复结论：先按 event_time 与稳定 event_id 排序，再执行预测后更新 memory。")  # 总结 TGN 数据入口和更新顺序门禁。

直接按文件顺序的负时间差记录=[(8, '用户-B', -1)]
按 event_time 排序后的负时间差记录=[]
修复结论：先按 event_time 与稳定 event_id 排序，再执行预测后更新 memory。


## 生产差距

线上 TGN 需要 watermark、迟到事件重放、外部 memory store、并发写冲突和快照恢复。长序列通常截断 BPTT，训练 memory 与服务 memory 还要做版本一致性。评估必须按时间切分并避免未来邻居泄漏，同时监控时间差分布、冷节点、状态陈旧度和回滚成本。

## 最小回归测试

In [7]:
assert len(events) >= 5  # 保证案例包含足够多的真实可读时间事件。
assert loss_trace[-1] < loss_trace[0]  # 保证完整事件流训练损失下降。
assert first_time_gradient > 0.0  # 保证可学习时间编码获得非零梯度。
assert tgn_accuracy > baseline_accuracy  # 保证时序记忆在同事件指标上超过金额规则。
assert tgn_accuracy == 1.0  # 保证教学模型拟合全部十二笔受控事件。
assert len(bad_negative_rows) > 0 and len(fixed_negative_rows) == 0  # 保证乱序失败与时间排序修复均可复现。
assert all(math.isfinite(value) for value in tgn_probabilities.tolist())  # 保证全部事件概率为有限数值。
print("回归测试通过：时间编码、记忆更新、事件预测和乱序门禁均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：时间编码、记忆更新、事件预测和乱序门禁均符合预期。
